# Counterexample Commons - Complete Colab Lab

Keyless Ollama-first launcher for the local-first Gradio research lab.

[Open in GitHub](https://github.com/error-wtf/counterexample-commons/tree/main)


## Scope and safety

This notebook launches the same Gradio app from `main`. In Google Colab it defaults to keyless Ollama Local mode inside the Colab VM.

It is designed for a fresh Google Colab runtime with no provider credentials loaded. Set `ENABLE_COLAB_OLLAMA = False` for finite-only public-demo mode.

Scientific boundary:

- Exact finite baseline calculations are locally executable.
- The finite rational mesh baseline is not Sawin's construction.
- Finite exact validation is not evidence for exponent n^1.014.
- OpenAI/Sawin asymptotic results are source-documented only here, not locally reproduced.
- API-key provider workflows are not part of this notebook.
- Ollama Local runs keyless AI inside the Colab VM; it cannot access your desktop `localhost`.


## Sources and theorem map

| Result | Source | Local status |
|---|---|---|
| Erdős unit-distance problem | Historical/source context | Source documented |
| OpenAI fixed-delta disproof | OpenAI announcement and proof PDF | SOURCE_DOCUMENTED; not locally reproduced |
| Original OpenAI proof explicit delta = 0.014 | Original proof does not provide this value | NOT_PROVIDED_BY_ORIGINAL_PROOF |
| Sawin explicit exponent n^1.014 | Will Sawin, arXiv:2605.20579 | SOURCE_DOCUMENTED; not locally reproduced |
| Finite rational mesh baseline | Repository exact validator | LOCALLY_REPRODUCED_EXACT finite baseline only |


## 1. Fresh runtime setup

Run this cell first. In Colab it clones `main`, installs the package and, by default, starts Ollama inside the Colab VM. Locally it only locates the current checkout for precheck use.

Set `ENABLE_COLAB_OLLAMA = False` below if you want finite-only public-demo mode instead of keyless Colab-VM AI.


In [ ]:
import os
import platform
import sys
import shutil
import subprocess
import time
import urllib.error
import urllib.request
from pathlib import Path

IN_COLAB = (
    "google.colab" in sys.modules
    or bool(os.environ.get("COLAB_RELEASE_TAG"))
    or bool(os.environ.get("COLAB_BACKEND_VERSION"))
)

REPO_URL = "https://github.com/error-wtf/counterexample-commons.git"
REPO_BRANCH = "main"
COLAB_REPO_DIR = Path("/content/counterexample-commons")
PROVIDER_ENV_VARS = [
    "OPENAI_API_KEY",
    "ANTHROPIC_API_KEY",
    "OPENROUTER_API_KEY",
    "GEMINI_API_KEY",
    "MISTRAL_API_KEY",
    "OLLAMA_API_KEY",
]
ENABLE_COLAB_OLLAMA = True
OLLAMA_MODEL = os.environ.get("OLLAMA_MODEL", "llama3.1:8b")
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
os.environ["PATH"] = (
    "/usr/local/bin:/usr/bin:/bin:" + os.environ.get("PATH", "")
)


def _ollama_linux_arch() -> str:
    machine = platform.machine().lower()
    if machine in {"x86_64", "amd64"}:
        return "amd64"
    if machine in {"aarch64", "arm64"}:
        return "arm64"
    raise RuntimeError(f"Unsupported Colab CPU architecture: {machine}")


def _run_checked(command: list[str]) -> None:
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    if result.returncode != 0:
        print(result.stdout[-4000:])
        raise subprocess.CalledProcessError(
            result.returncode,
            command,
            output=result.stdout,
        )


def _install_zstd_if_needed() -> None:
    if shutil.which("zstd") is not None:
        return
    print("Installing zstd for Ollama archive extraction...")
    _run_checked(["apt-get", "update"])
    _run_checked(["apt-get", "install", "-y", "zstd"])


def _download_and_extract_ollama() -> None:
    arch = _ollama_linux_arch()
    install_root = Path("/usr/local")
    archive_dir = Path("/content/ollama-download")
    archive_dir.mkdir(parents=True, exist_ok=True)
    archive_attempts = [
        (
            f"https://ollama.com/download/ollama-linux-{arch}.tar.zst",
            archive_dir / f"ollama-linux-{arch}.tar.zst",
            ["tar", "--zstd", "-xf"],
        ),
        (
            f"https://ollama.com/download/ollama-linux-{arch}.tgz",
            archive_dir / f"ollama-linux-{arch}.tgz",
            ["tar", "-xzf"],
        ),
    ]
    errors = []
    for url, archive, tar_prefix in archive_attempts:
        try:
            if archive.suffix == ".zst":
                _install_zstd_if_needed()
            print(f"Downloading Ollama archive: {url}")
            _run_checked(["curl", "-fL", url, "-o", str(archive)])
            print(f"Extracting Ollama into {install_root}...")
            _run_checked([*tar_prefix, str(archive), "-C", str(install_root)])
            if shutil.which("ollama") or Path("/usr/local/bin/ollama").exists():
                return
            raise RuntimeError("Ollama binary was not found after extraction.")
        except Exception as exc:
            errors.append(f"{url}: {exc.__class__.__name__}: {exc}")
            print(f"Ollama archive attempt failed: {errors[-1]}")
    raise RuntimeError(
        "Could not install Ollama in this Colab runtime. Attempts:\n"
        + "\n".join(errors)
    )


def _ollama_server_ready() -> bool:
    try:
        with urllib.request.urlopen(
            OLLAMA_BASE_URL.rstrip("/") + "/api/tags",
            timeout=3,
        ) as response:
            return 200 <= response.status < 300
    except (urllib.error.URLError, TimeoutError, OSError):
        return False


def _wait_for_ollama_server() -> None:
    deadline = time.time() + 90
    last_error = None
    while time.time() < deadline:
        if _ollama_server_ready():
            return
        last_error = RuntimeError("Ollama server not ready yet")
        time.sleep(2)
    raise RuntimeError(
        "Ollama did not start inside the Colab runtime. "
        f"Last error: {last_error.__class__.__name__ if last_error else 'unknown'}"
    )


def _install_and_start_colab_ollama() -> None:
    if not IN_COLAB:
        raise RuntimeError("ENABLE_COLAB_OLLAMA is only intended for Colab.")
    if shutil.which("ollama") is None:
        print("Installing Ollama inside the Colab VM from official archives...")
        _download_and_extract_ollama()
    else:
        print("Ollama binary already present in this runtime.")
    if not _ollama_server_ready():
        print("Starting Ollama server inside the Colab VM...")
        log_file = open("/content/ollama_server.log", "ab")
        subprocess.Popen(
            ["ollama", "serve"],
            stdout=log_file,
            stderr=subprocess.STDOUT,
        )
    else:
        print("Ollama server is already running inside the Colab VM.")
    os.environ["OLLAMA_BASE_URL"] = OLLAMA_BASE_URL
    os.environ["OLLAMA_MODEL"] = OLLAMA_MODEL
    _wait_for_ollama_server()
    print(f"Ensuring Ollama model is available: {OLLAMA_MODEL}")
    subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
    print("Colab Ollama Local mode: READY")

loaded_provider_vars = [
    name for name in PROVIDER_ENV_VARS
    if os.environ.get(name)
]
if loaded_provider_vars:
    raise RuntimeError(
        "Provider credential environment variables are present. "
        "Do not launch this keyless Colab notebook from a credentialed runtime. "
        "Start a fresh runtime or remove credentials first."
    )

if IN_COLAB:
    if not COLAB_REPO_DIR.exists():
        subprocess.run(
            [
                "git", "clone", "--depth", "1",
                "--branch", REPO_BRANCH, REPO_URL,
                str(COLAB_REPO_DIR),
            ],
            check=True,
        )
    elif (COLAB_REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "-C", str(COLAB_REPO_DIR), "fetch", "origin", REPO_BRANCH],
            check=True,
        )
        subprocess.run(
            [
                "git", "-C", str(COLAB_REPO_DIR), "checkout", "-B",
                REPO_BRANCH, f"origin/{REPO_BRANCH}",
            ],
            check=True,
        )
    else:
        raise RuntimeError(
            f"{COLAB_REPO_DIR} exists but is not a Git checkout. "
            "Move it aside or restart the runtime."
        )

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(COLAB_REPO_DIR)],
        check=True,
    )
    repo_root = COLAB_REPO_DIR
    if ENABLE_COLAB_OLLAMA:
        _install_and_start_colab_ollama()
        runtime_label = "GOOGLE COLAB PRIVATE OLLAMA LOCAL MODE"
    else:
        runtime_label = "GOOGLE COLAB PUBLIC NO-KEY DEMO"
else:
    start = Path.cwd().resolve()
    repo_root = next(
        (
            p for p in [start, *start.parents]
            if (p / "counterexample_commons").is_dir()
            and (p / "pyproject.toml").is_file()
        ),
        None,
    )
    if repo_root is None:
        raise RuntimeError("Repository root not found for local precheck.")
    runtime_label = "LOCAL PRECHECK ONLY - NOT GOOGLE COLAB EVIDENCE"

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import counterexample_commons

print(runtime_label)
print(f"Repository branch: {REPO_BRANCH}")
print(f"Package version: {counterexample_commons.__version__}")
if ENABLE_COLAB_OLLAMA and IN_COLAB:
    print(f"Ollama Local: enabled inside Colab VM with {OLLAMA_MODEL}")
else:
    print("Provider credentials: not configured for this public demo")


## 2. Exact finite self-test

This cell verifies the exact rational validator and the finite rational mesh baseline before the UI is launched.


In [ ]:
from sympy import Rational
from counterexample_commons.validators import (
    count_unit_edges_exact,
    validate_grid_configuration,
    validate_line_configuration,
    validate_custom_configuration,
)
from case_studies.erdos_unit_distance_2026.rational_mesh_baseline import (
    verify_rational_mesh,
)

pair_points = [
    (Rational(0), Rational(0)),
    (Rational(3, 5), Rational(4, 5)),
]
pair_count, pair_edges = count_unit_edges_exact(pair_points)
assert pair_count == 1
assert pair_edges == [(0, 1)]

line = validate_line_configuration(5)
assert line["pass"] is True

grid = validate_grid_configuration(5)
assert grid["pass"] is True

rational_example = validate_custom_configuration([
    ("0", "0"),
    ("3/5", "4/5"),
])
assert rational_example["actual_edges"] == 1

mesh = verify_rational_mesh(10)
assert mesh["n"] == 121
assert mesh["unit_edges"] == 82

print("Exact validator self-test: PASS")
print("Rational 3/5-4/5 example: 1 exact unit-distance edge")
print("Finite Rational Mesh Baseline m=10: 121 points")
print("Finite Rational Mesh Baseline m=10: 82 exactly validated unit-distance edges")
print("Finite rational mesh baseline - not Sawin's construction.")
print("Finite exact validation only; not evidence for exponent n^1.014.")


## 3. Launch the complete Gradio lab

This starts the app in `colab-private` mode by default when running in Colab with `ENABLE_COLAB_OLLAMA = True`; otherwise it uses finite-only `colab-public-demo` mode.

The Ollama path uses only the Colab VM's local Ollama server and no API key. The finite-only fallback exposes exact baselines, explorer, visualisation, read-only claims and finite exports.


In [ ]:
from app.main import build_app
from counterexample_commons.config import AppMode

launch_mode = (
    AppMode.COLAB_PRIVATE
    if ENABLE_COLAB_OLLAMA and IN_COLAB
    else AppMode.COLAB_PUBLIC_DEMO
)

print(f"Launching Counterexample Commons in {launch_mode.value} mode.")
print("No API keys are stored in this notebook. Claim registry read-only.")
if launch_mode is AppMode.COLAB_PRIVATE:
    print("Ollama Local is expected inside the Colab VM; no API key required.")
else:
    print("AI/provider live workflows are disabled in this public Colab demo.")

app = build_app(launch_mode)

if IN_COLAB:
    app.queue().launch(share=True, debug=False, show_error=True)
else:
    print("Local precheck: app built successfully.")
    print("In Google Colab this cell launches with share=True.")


## 4. Keyless Ollama Local workflow

This notebook is keyless by default. Do not paste provider keys into notebook cells.

In Colab, `ENABLE_COLAB_OLLAMA = True` starts Ollama inside the Colab VM and launches the app in `colab-private` mode. Set it to `False` for finite-only public-demo mode. AI-generated candidates remain finite hypotheses until independently checked by the exact validator.


## 5. Runtime status

Expected default Colab status:

```text
COLAB_MODE: colab-private when ENABLE_COLAB_OLLAMA = True
OLLAMA_MODE: Colab VM local Ollama, no API key
PROVIDER_KEYS: not configured
LIVE_PROVIDER_CALLS: local Ollama only; API-key providers disabled
CLAIM_REGISTRY: read-only
SAWIN_N_1_014: SOURCE_DOCUMENTED; not locally reproduced
FINITE_RATIONAL_MESH: LOCALLY_REPRODUCED_EXACT finite baseline only
```
